# Multi-Modal Emergency Incident Severity Prediction (Model V3)

## Objective
This notebook expands the system to handle **Fire, Medical, Crime, and Traffic** incidents. 
Since a unified public dataset is unavailable, we **simulate** Fire, Medical, and Crime data by augmenting the real Traffic dataset. 
The goal is to demonstrate a **Prioritization System** that can rank diverse emergencies based on predicted severity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Set seeds for reproducibility
np.random.seed(42)

: 

## 1. Load Real Traffic Data
We use a subset of the US Accidents dataset as the base for our "Traffic" incidents.

In [ ]:
# Load Traffic Data
df_traffic = pd.read_csv('../data/US_Accidents_March23.csv', nrows=50000)
df_traffic['Start_Time'] = pd.to_datetime(df_traffic['Start_Time'], errors='coerce')

# Map Traffic Schema to Unified Schema
traffic_data = pd.DataFrame({
    'Incident_Type': 'Traffic',
    'Subtype': df_traffic['Description'].str.split().str[0], # Rough proxy
    'Start_Time': df_traffic['Start_Time'],
    'Lat': df_traffic['Start_Lat'],
    'Lng': df_traffic['Start_Lng'],
    'City': df_traffic['City'],
    'Temperature(F)': df_traffic['Temperature(F)'],
    'Humidity(%)': df_traffic['Humidity(%)'],
    'Hour': df_traffic['Start_Time'].dt.hour,
    'Month': df_traffic['Start_Time'].dt.month,
    'DayOfWeek': df_traffic['Start_Time'].dt.dayofweek,
    'Severity_Score': df_traffic['Severity'] + 1 # Shift 1-4 to 2-5 (Traffic is rarely a 10)
})

# Clean Traffic Data
traffic_data['Subtype'] = traffic_data['Subtype'].fillna('Accident')
traffic_data = traffic_data.dropna(subset=['Lat', 'Lng'])

print(f"Traffic Samples: {len(traffic_data)}")

## 2. Simulate Fire, Medical, and Crime Data
We generate synthetic data to represent other agencies. We use the location/weather context from traffic data to make it realistic (e.g., fires are more likely in hot/dry weather, crime at night).

In [ ]:
def generate_synthetic_incidents(base_df, n_samples, incident_type, severity_dist, subtypes):
    # Sample random rows from base_df to get realistic locations and times
    sample = base_df.sample(n_samples, replace=True).copy().reset_index(drop=True)
    
    sample['Incident_Type'] = incident_type
    sample['Subtype'] = np.random.choice(subtypes, size=n_samples)
    
    # Generate Severity based on type logic
    # We assume 'severity_dist' is a tuple (mean, std)
    mean_sev, std_sev = severity_dist
    sample['Severity_Score'] = np.random.normal(mean_sev, std_sev, size=n_samples)
    sample['Severity_Score'] = sample['Severity_Score'].clip(1, 10).round().astype(int)
    
    # Add some correlation logic (Bonus)
    if incident_type == 'Fire':
        # Fires slightly more severe in low humidity
        sample.loc[sample['Humidity(%)'] < 30, 'Severity_Score'] += 1
    if incident_type == 'Crime':
        # Crimes slightly more severe at night
        sample.loc[(sample['Hour'] > 22) | (sample['Hour'] < 5), 'Severity_Score'] += 1
        
    return sample

# Generate Fire Data (Rare, High Severity)
fire_data = generate_synthetic_incidents(
    traffic_data, 
    n_samples=5000, 
    incident_type='Fire', 
    severity_dist=(7, 2), 
    subtypes=['Structure Fire', 'Brush Fire', 'Vehicle Fire', 'Trash Fire']
)

# Generate Medical Data (Frequent, Variable Severity)
medical_data = generate_synthetic_incidents(
    traffic_data, 
    n_samples=20000, 
    incident_type='Medical', 
    severity_dist=(5, 2.5), 
    subtypes=['Cardiac Arrest', 'Respiratory Distress', 'Trauma', 'Fall', 'Overdose']
)

# Generate Crime Data (Frequent, Variable Severity)
crime_data = generate_synthetic_incidents(
    traffic_data, 
    n_samples=15000, 
    incident_type='Crime', 
    severity_dist=(4, 3), 
    subtypes=['Robbery', 'Burglary', 'Assault', 'Domestic', 'Theft']
)

# Merge All Datasets
df_unified = pd.concat([traffic_data, fire_data, medical_data, crime_data]).reset_index(drop=True)
df_unified = df_unified.sample(frac=1, random_state=42).reset_index(drop=True) # Shuffle

print("Unified Dataset Distribution:")
print(df_unified['Incident_Type'].value_counts())
print("\nSeverity Stats per Type:")
print(df_unified.groupby('Incident_Type')['Severity_Score'].mean())

## 3. Train Multi-Modal Severity Predictor
We use `HistGradientBoostingRegressor` to predict a continuous severity score (1-10). The model interprets `Incident_Type` as a critical feature.

In [ ]:
# Features & Target
X = df_unified[['Incident_Type', 'Subtype', 'Lat', 'Lng', 'City', 'Hour', 'Month', 'DayOfWeek', 'Temperature(F)', 'Humidity(%)']]
y = df_unified['Severity_Score']

# Pipeline
categorical_features = ['Incident_Type', 'Subtype', 'City']
numerical_features = ['Lat', 'Lng', 'Hour', 'Month', 'DayOfWeek', 'Temperature(F)', 'Humidity(%)']

preprocessor = ColumnTransformer(
    transformers=[
        # TargetEncoder is great for high cardinality (City) and categorical features
        ('cat', TargetEncoder(target_type='continuous'), categorical_features),
        ('num', 'passthrough', numerical_features)
    ],
    verbose_feature_names_out=False
).set_output(transform='pandas')

model_v3 = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(random_state=42))
])

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train
print("Training Multi-Modal Model...")
model_v3.fit(X_train, y_train)
print("Training Complete.")

# Evaluate
y_pred = model_v3.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.2f}")

## 4. Prioritization Demo (Ranking Scenario)
This is the core value proposition. We feed the model three simultaneous incidents and ask it to rank them.

In [ ]:
# Create a live scenario
live_incidents = pd.DataFrame([
    {
        'Incident_Type': 'Traffic', 
        'Subtype': 'Accident', 
        'Lat': 40.7128, 'Lng': -74.0060, 'City': 'New York', 
        'Hour': 14, 'Month': 5, 'DayOfWeek': 2, 
        'Temperature(F)': 75, 'Humidity(%)': 50
    },
    {
        'Incident_Type': 'Fire', 
        'Subtype': 'Structure Fire', 
        'Lat': 40.7128, 'Lng': -74.0060, 'City': 'New York', 
        'Hour': 14, 'Month': 5, 'DayOfWeek': 2, 
        'Temperature(F)': 75, 'Humidity(%)': 50
    },
    {
        'Incident_Type': 'Medical', 
        'Subtype': 'Trauma',  
        'Lat': 40.7128, 'Lng': -74.0060, 'City': 'New York', 
        'Hour': 14, 'Month': 5, 'DayOfWeek': 2, 
        'Temperature(F)': 75, 'Humidity(%)': 50
    },
    {
        'Incident_Type': 'Crime',
        'Subtype': 'Theft',
        'Lat': 40.7128, 'Lng': -74.0060, 'City': 'New York', 
        'Hour': 14, 'Month': 5, 'DayOfWeek': 2, 
        'Temperature(F)': 75, 'Humidity(%)': 50
    }
])

print("Analyzing Incoming Incidents...")

# Predict Scores
live_incidents['Predicted_Score'] = model_v3.predict(live_incidents)

# Rank
ranked_incidents = live_incidents.sort_values('Predicted_Score', ascending=False).reset_index(drop=True)
ranked_incidents['Rank'] = ranked_incidents.index + 1

# Display
print("\n--- RECOMMENDED RESPONSE PRIORITY ---")
print(ranked_incidents[['Rank', 'Incident_Type', 'Subtype', 'Predicted_Score']])